# Entscheidungsbaum für Regression

Ein Regressionsbaum sucht Gruppen mit ähnlichen Zielwerten. Im Blatt sagt er meist deren Mittelwert vorher - dadurch entsteht eine Treppenfunktion.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.tree import DecisionTreeRegressor, plot_tree
rng=np.random.default_rng(42)
X=np.linspace(-3,3,500).reshape(-1,1)
signal=1.4*np.sin(1.8*X[:,0])+.3*X[:,0]**2
y=signal+rng.normal(0,.35,len(X))
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=42)
plt.scatter(X_train[:,0],y_train,s=16,alpha=.5); plt.plot(X[:,0],signal,color="black"); plt.show()

## Hyperparameter

`criterion` bewertet die Streuung (`squared_error`, `absolute_error`, `friedman_mse`, `poisson`). `max_depth`, `min_samples_split`, `min_samples_leaf`, `max_leaf_nodes` und `ccp_alpha` begrenzen die Komplexität. Größere Blätter glätten die Vorhersage besonders direkt.

In [ ]:
def regmetriken(name,m):
    p=m.predict(X_test)
    return {"Modell":name,"MAE":mean_absolute_error(y_test,p),"RMSE":mean_squared_error(y_test,p)**.5,"R2":r2_score(y_test,p)}
raster=np.linspace(-3,3,600).reshape(-1,1)
modelle={"Tiefe 2":DecisionTreeRegressor(max_depth=2,random_state=42),"Tiefe 5":DecisionTreeRegressor(max_depth=5,random_state=42),"Ungebremst":DecisionTreeRegressor(random_state=42)}
fig,ax=plt.subplots(1,3,figsize=(15,4),sharey=True)
for a,(n,m) in zip(ax,modelle.items()):
    m.fit(X_train,y_train); a.scatter(X_train[:,0],y_train,s=8,alpha=.25); a.plot(raster[:,0],m.predict(raster),color="tomato"); a.set_title(n)
plt.show(); display(pd.DataFrame([regmetriken(n,m) for n,m in modelle.items()]).set_index("Modell").round(3))

werte=[]
for d in range(1,18):
    m=DecisionTreeRegressor(max_depth=d,random_state=42).fit(X_train,y_train)
    werte.append({"Tiefe":d,"RMSE Training":mean_squared_error(y_train,m.predict(X_train))**.5,"RMSE Test":mean_squared_error(y_test,m.predict(X_test))**.5})
pd.DataFrame(werte).plot(x="Tiefe",marker="o"); plt.grid(alpha=.3); plt.show()

## Kleine Grid Search und Vergleich

`neg_root_mean_squared_error` ist negativ, weil scikit-learn Scores maximiert.

In [ ]:
grid={"max_depth":[2,3,5,8,None],"min_samples_leaf":[1,5,15],"ccp_alpha":[0,.002]}
suche=GridSearchCV(DecisionTreeRegressor(random_state=42),grid,scoring="neg_root_mean_squared_error",cv=5,n_jobs=-1).fit(X_train,y_train)
dummy=DummyRegressor().fit(X_train,y_train)
print("Beste Parameter:",suche.best_params_,"CV-RMSE:",round(-suche.best_score_,3))
display(pd.DataFrame([regmetriken("Mittelwert",dummy),regmetriken("Ungebremst",modelle["Ungebremst"]),regmetriken("Grid Search",suche.best_estimator_)]).set_index("Modell").round(3))
plt.scatter(X_test[:,0],y_test,s=18,alpha=.4); plt.plot(raster[:,0],suche.predict(raster),color="tomato"); plt.plot(X[:,0],signal,"k--"); plt.show()
plt.figure(figsize=(14,6)); plot_tree(suche.best_estimator_,max_depth=3,filled=True,rounded=True,feature_names=["x"]); plt.show()